# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process a rich clinical oncology dataset using the [mlcroissant](https://github.com/mlcommons/croissant-python) library.

### Dataset Source
The dataset source is defined by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata: title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Authors: {getattr(dataset.metadata, 'author', 'N/A')}")

## 2. Data Overview
Let's review available record sets and their fields using their unique `@id` identifiers.

We list all record sets, then for each record set, we print its fields and show their `@id`s.

In [ ]:
# Find all record set @ids in the dataset metadata

record_sets = []
if hasattr(dataset.metadata, 'recordSet'):
    if isinstance(dataset.metadata.recordSet, list):
        record_sets = dataset.metadata.recordSet
    elif dataset.metadata.recordSet:
        record_sets = [dataset.metadata.recordSet]
    else:
        record_sets = []

# record_sets may be empty if record sets are not defined in top-level metadata, try to infer using dataset API
if not record_sets:
    record_sets = dataset.record_sets()
    print(f"Discovered record sets via mlcroissant: {record_sets}")

print("Dataset Record Sets (by @id):")
for rid in record_sets:
    print(f"- {rid}")

# For each record set, print available fields and their @ids
print("\nRecord set fields overview:")
for rid in record_sets:
    fields = dataset.fields(record_set=rid)
    print(f"\nRecord Set: {rid}")
    for field in fields:
        print(f"  Field @id: {field['@id']} | Name: {field['name']} | DataType: {field.get('dataType', '-')}")

## 3. Data Extraction
Let's load all data from each record set into a Pandas DataFrame. We'll use only the `@id` values to reference record sets and fields, as per best practice.

We'll examine the columns for each record set and preview the top rows.

In [ ]:
# For this dataset, let's get all available record sets via API
record_set_ids = dataset.record_sets()
print(f"Record sets discovered: {record_set_ids}")

dataframes = {}
for rid in record_set_ids:
    # Load records for each record set
    print(f"\nExtracting {rid}")
    rows = list(dataset.records(record_set=rid))
    if not rows:
        print(f"  No records found for {rid}.")
        continue
    df = pd.DataFrame(rows)
    dataframes[rid] = df
    print(f"  Columns: {df.columns.tolist()}")
    display(df.head(3))

# If no record sets were found, notify user
if not dataframes:
    print("No record sets found with records to extract.")

## 4. Exploratory Data Analysis (EDA)
Now that we have the data in DataFrames, let's select a primary record set for demonstration. We'll:
- Filter rows based on a chosen numeric variable (by field `@id`)
- Normalize this numeric field
- Optionally group by a categorical variable (by field `@id`)

*Note: We'll use the first available record set and attempt to automatically select a numeric field and a group-by field based on type and name. Adjust as needed for your own analysis.*

In [ ]:
from pandas.api.types import is_numeric_dtype

if dataframes:
    # Select the first available record set and DataFrame
    selected_rid = next(iter(dataframes.keys()))
    df = dataframes[selected_rid].copy()
    print(f"Using record set: {selected_rid}")

    # Attempt to pick a numeric field (@id) automatically
    numeric_field_id = None
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Numeric field selected: {numeric_field_id}")

        # Filter: keep records where value > 10 (arbitrary demo threshold)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize the numeric column
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"First normalized values for {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try group by a likely categorical field (pick first non-numeric field)
        group_field_id = None
        for col in df.columns:
            if not is_numeric_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped.head())
        else:
            print("No suitable group-by field available.")
else:
    print("No data available for EDA.")

## 5. Visualization
Let's visualize the distribution of our selected numeric variable, and, if available, compare it across the chosen group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if dataframes and numeric_field_id:
    fig, axs = plt.subplots(1, 2 if group_field_id else 1, figsize=(10, 4))
    if not isinstance(axs, (list, np.ndarray)):
        axs = [axs]

    # Plot distribution of the numeric field
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, ax=axs[0])
    axs[0].set_title(f"Distribution of {numeric_field_id}")

    # Optionally, boxplot by group
    if group_field_id and group_field_id in df.columns:
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, ax=axs[1])
        axs[1].set_title(f"{numeric_field_id} by {group_field_id}")
        axs[1].set_xticklabels(axs[1].get_xticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    plt.show()
else:
    print("No sufficient data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer dataset using the `mlcroissant` library. We:

- Loaded the dataset using a Croissant schema URL
- Explored available record sets and fields using their unique `@id`
- Extracted records into Pandas DataFrames
- Applied basic EDA and data processing tasks: filtered, normalized, grouped records
- Visualized a numeric variable and compared across a group field

This workflow demonstrates how FAIR data and the Croissant standard enable easy, transparent, and reproducible clinical data analysis.